# 06 部署：把模型变成服务

> 前置：`02-baselines-tabular`（训练）、`05-evaluation-error-analysis`。
> 目标：模型训练完 ≠ 能用。本课走通 **导出 → 加载验证 → 预测接口 → 性能测量** 的部署最小闭环，并给出 FastAPI 在线服务参考实现。

## 部署的四个层次

1. **文件导出**：模型序列化到磁盘（joblib/pickle/ONNX），可移植、可版本化；
2. **加载验证**：新进程加载，预测必须与训练时一致（部署最常见翻车点）；
3. **接口化**：把"原始输入 → 预测"封装成标准函数/HTTP 接口；
4. **服务化**：在线 API（延迟敏感）或批处理（吞吐优先），加监控（07 课）。

顺序很重要：**先保证 1→2 一致，再谈 3→4 的性能**。

In [1]:
# 本模块通用导入（全部 CPU 即可运行）
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

# 中文字体兼容（Windows / macOS）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

numpy 2.5.2 | pandas 3.0.5 | sklearn 1.9.0


In [2]:
import joblib, time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import io, urllib.request

TITANIC_URLS = [
    "https://cdn.jsdelivr.net/gh/datasciencedojo/datasets@master/titanic.csv",
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
]

def load_titanic():
    """本地缓存优先，CDN 在线下载兜底；全部失败返回 None（触发下方 sklearn 回退）。"""
    for p in (os.path.join("..", "titanic.csv"), "titanic.csv"):
        if os.path.exists(p):
            return pd.read_csv(p)
    for url in TITANIC_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as r:
                data = r.read()
            with open(os.path.join("..", "titanic.csv"), "wb") as f:
                f.write(data)
            return pd.read_csv(io.BytesIO(data))
        except Exception:
            continue
    return None

try:
    df = load_titanic()
    d = df.set_index("PassengerId")
    y = d["Survived"].astype(int); d = d.drop(columns=["Survived"])
    d["Age"] = d["Age"].fillna(d["Age"].median())
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna(d["Embarked"].mode()[0])
    d = pd.get_dummies(d, columns=["Sex", "Embarked"], drop_first=True)
    X = d[["Pclass", "Age", "Fare", "SibSp", "Parch", "Sex_male", "Embarked_Q", "Embarked_S"]]
except Exception:
    from sklearn.datasets import load_breast_cancer
    Xb, yb = load_breast_cancer(return_X_y=True, as_frame=True)
    X, y = Xb, yb

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr, y_tr)
print("模型：", type(model).__name__, " 训练完成")

# 1) 导出
joblib.dump(model, "model.joblib")
print("已导出 model.joblib（", os.path.getsize("model.joblib") // 1024, "KB ）")

# 2) 加载验证：新对象预测必须与内存模型完全一致
loaded = joblib.load("model.joblib")
same = np.array_equal(model.predict(X_te), loaded.predict(X_te))
print("导出→加载 预测一致性：", same)
assert same, "导出加载不一致，禁止上线！"

模型： RandomForestClassifier  训练完成
已导出 model.joblib（ 4727 KB ）
导出→加载 预测一致性： True


## 导出格式怎么选

| 格式 | 适用 | 特点 |
|------|------|------|
| joblib / pickle | sklearn 模型 | 最快、Python 内最稳；跨语言不行 |
| ONNX | 跨平台/跨语言 | 可转 TensorRT/ONNX Runtime，生产主流 |
| TorchScript | PyTorch 模型 | torch 官方序列化，C++ 部署 |
| safetensors | LLM 权重 | 安全、无 pickle 反序列化漏洞 |

**原则**：模型和**特征工程逻辑**要一起版本化——光有 `model.joblib` 没有 `featurize()`，任何新数据都喂不进去。

In [3]:
# 3) 接口化：原始输入 -> 特征 -> 预测（生产环境的标准函数，必须与训练时完全一致）
FEATURES = ["Pclass", "Age", "Fare", "SibSp", "Parch", "Sex_male", "Embarked_Q", "Embarked_S"]

def featurize(pclass, sex, age, fare, sibsp, parch, embarked):
    """把一条原始乘客记录转成训练时的特征向量。"""
    return pd.DataFrame([{
        "Pclass": pclass, "Age": age, "Fare": fare, "SibSp": sibsp, "Parch": parch,
        "Sex_male": 1 if sex == "male" else 0,
        "Embarked_Q": 1 if embarked == "Q" else 0,
        "Embarked_S": 1 if embarked == "S" else 0,
    }], columns=FEATURES)

def predict_survival(pclass, sex, age, fare, sibsp, parch, embarked):
    """在线预测接口：原始字段 -> 生存概率与预测。"""
    x = featurize(pclass, sex, age, fare, sibsp, parch, embarked)
    prob = float(loaded.predict_proba(x)[0, 1])
    return {"survive_prob": round(prob, 4), "survive": int(prob >= 0.5)}

print("接口测试（头等舱年轻女性）：", predict_survival(1, "female", 29, 80, 0, 0, "C"))
print("接口测试（三等舱老年男性）：", predict_survival(3, "male", 70, 8, 0, 0, "S"))

# 批量推理
batch = X_te.iloc[:100]
t0 = time.perf_counter()
probs = loaded.predict_proba(batch)[:, 1]
dt = time.perf_counter() - t0
print(f"批量推理 100 条：{dt*1000:.1f} ms，单条平均 {dt*1000/100:.2f} ms")

接口测试（头等舱年轻女性）： {'survive_prob': 0.98, 'survive': 1}
接口测试（三等舱老年男性）： {'survive_prob': 0.04, 'survive': 0}
批量推理 100 条：23.2 ms，单条平均 0.23 ms


## FastAPI 在线服务（参考实现）

```python
# app.py —— 生产环境把它放到独立服务进程，本课不启动以免阻塞 notebook
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, pandas as pd

model = joblib.load("model.joblib")
app = FastAPI()

class Passenger(BaseModel):
    Pclass: int; Age: float; Fare: float
    SibSp: int; Parch: int
    Sex_male: int; Embarked_Q: int; Embarked_S: int

@app.post("/predict")
def predict(p: Passenger):
    df = pd.DataFrame([p.model_dump()])
    prob = float(model.predict_proba(df)[0, 1])
    return {"survive_prob": round(prob, 4), "survive": int(prob >= 0.5)}
```

启动：`uvicorn app:app --reload`，然后 `POST /predict` 发 JSON 即可。要点：**接口层只负责序列化，特征工程在接口函数里复用同一份代码**。

In [4]:
# 4) 性能测量：延迟与吞吐是 SLA 的语言
single = X_te.iloc[0:1]
times = []
for _ in range(300):
    t0 = time.perf_counter()
    loaded.predict_proba(single)
    times.append(time.perf_counter() - t0)
times = np.array(times) * 1000
print(f"单条推理延迟：P50 = {np.median(times):.2f} ms   P95 = {np.percentile(times, 95):.2f} ms")

t0 = time.perf_counter()
for _ in range(1000):
    loaded.predict_proba(X_te.iloc[:100])
total = time.perf_counter() - t0
print(f"吞吐：{1000 * 100 / total:.0f} 条/秒（CPU 单进程）")

单条推理延迟：P50 = 19.75 ms   P95 = 22.56 ms


吞吐：3984 条/秒（CPU 单进程）


## 上线注意

- **版本管理**：模型文件 + 特征工程代码 + 训练数据版本，三者绑定（如 `model_v3.1_joblib` + git tag）；
- **A/B 测试**：新旧模型各接一部分流量，用线上指标（而非离线分数）决定是否全量；
- **回滚**：新模型出问题，能一键切回旧版本——**模型文件本身要可回滚**；
- **监控**：上线后接 07 课的漂移检测，模型会变老。

> 部署的黄金法则：**离线评估能骗人，线上指标才作数；能回滚的部署才叫部署。**

## 课后练习

1. **接口练习**：把本课 `featurize` + `predict_survival` 改成对 10 条真实记录批量预测，并验证与 `model.joblib` 直接预测结果一致（一致性是部署的生命线）。
2. **本地 API**：按参考实现安装 `fastapi` + `uvicorn`，在本地把模型起成服务，用 `requests` 发一条 JSON 测试，返回格式与 `predict_survival` 一致。
3. **延迟预算练习**：假设业务要求 P95 < 50ms，测出当前延迟，提出两个优化方向（如特征裁剪、模型精简、批处理）并说明各自代价。
4. **Kaggle 关联**：Kaggle 竞赛的"提交文件"本质就是批处理部署——你提交的 `submission.csv` 是离线推理产物；想想在线推理和它的差异（延迟、数据可用性、特征来源）。